# 15장. 신경망 예측의 보정과 불확실성

신경망이 낸 확률이 실제로 맞는 비율과 같은지를 재고, 온도 조정과 시드 앙상블로 고친다. 학습 자료에 없던 종류의 입력에 내는 확률을 확인하고, 지표의 신뢰구간에 평가 자료의 변동과 시드의 변동을 함께 넣는다.

**준비:** 9장의 파일 `train-images-idx3-ubyte.gz`, `train-labels-idx1-ubyte.gz`(https://github.com/zalandoresearch/fashion-mnist)와 1장의 전력수요 파일 두 개(2023년, 2025년), 기온 파일 두 개(2023년, 2025년)를 왼쪽 파일 창에 올린 뒤 위에서부터 차례로 실행한다. Fashion-MNIST의 시험용 파일은 16장에서 쓴다. 끝까지 실행하는 데 7분쯤 걸린다.

## 15.1 신경망의 보정 곡선과 과신

In [ ]:
import glob, gzip, math, time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn

torch.set_num_threads(2)

def read_idx(path, offset):
    """idx 형식의 파일을 읽는다. 앞의 offset바이트는 머리말이다."""
    with gzip.open(path) as f:
        return np.frombuffer(f.read(), np.uint8, offset=offset)

images = read_idx("train-images-idx3-ubyte.gz", 16).reshape(-1, 28, 28)
labels = read_idx("train-labels-idx1-ubyte.gz", 8).astype(np.int64)
names = ["티셔츠", "바지", "스웨터", "드레스", "코트",
         "샌들", "셔츠", "운동화", "가방", "앵클부츠"]
order = np.random.default_rng(0).permutation(60000)           # 9장과 같은 나누기
train_rows, valid_rows = order[:6000], order[50000:]

def tensors(rows, mean=None, std=None):
    """고른 이미지를 (장수, 1, 28, 28) 모양의 실수 텐서로 바꾸고 표준화한다."""
    x = torch.tensor(images[rows] / 255, dtype=torch.float32)[:, None]
    if mean is None:
        mean, std = x.mean().item(), x.std().item()
    return (x - mean) / std, torch.tensor(labels[rows]), mean, std

X_tr, y_tr, pixel_mean, pixel_std = tensors(train_rows)
X_va, y_va, _, _ = tensors(valid_rows, pixel_mean, pixel_std)
FIT, TEST = slice(0, 5000), slice(5000, 10000)     # 검증 자료의 앞 절반과 뒤 절반
print(X_tr.shape, X_va.shape, len(y_va[FIT]), len(y_va[TEST]))

In [ ]:
def make_cnn(classes=10):
    """9.3절의 모델: 합성곱(16) - 풀링 - 합성곱(32) - 풀링 - 선형."""
    return nn.Sequential(
        nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(), nn.Linear(32 * 7 * 7, classes))

def scores_of(net, x):
    """소프트맥스에 넣기 전의 점수. 1,000장씩 나눠 계산한다."""
    net.eval()
    with torch.no_grad():
        return torch.cat([net(x[k:k + 1000]) for k in range(0, len(x), 1000)])

def fit(net, x, y, seed, epochs, watch=None):
    """9장의 학습 방법: Adam 0.001, 배치 크기 64. 에폭마다 watch를 부른다."""
    optimizer = torch.optim.Adam(net.parameters(), lr=0.001)
    shuffler = torch.Generator().manual_seed(seed)
    for epoch in range(epochs):
        net.train()
        mixed = torch.randperm(len(x), generator=shuffler)
        for start in range(0, len(x), 64):
            rows = mixed[start:start + 64]
            optimizer.zero_grad()
            F.cross_entropy(net(x[rows]), y[rows]).backward()
            optimizer.step()
        if watch is not None:
            watch(epoch + 1, net)
    return net

def accuracy(p, y):
    return (p.argmax(dim=1) == y).float().mean().item() * 100

def nll(p, y):
    """정답 클래스에 준 확률의 음의 로그를 평균한 값(교차 엔트로피 손실)."""
    return -torch.log(p[torch.arange(len(y)), y].clamp_min(1e-12)).mean().item()

def brier(p, y):
    """클래스마다 (확률 - 정답 표시)를 제곱해 더한 값의 평균."""
    return ((p - F.one_hot(y, p.shape[1]).float()) ** 2).sum(dim=1).mean().item()

def bins_of(p, y, bins=10):
    """확신을 0.1 간격의 구간으로 나눠 (장수, 평균 확신, 맞힌 비율)을 구한다."""
    sure, guess = p.max(dim=1)
    right = (guess == y).float()
    where = torch.clamp((sure * bins).long(), max=bins - 1)
    made = []
    for b in range(bins):
        inside = where == b
        if inside.any():
            made.append((b / bins, int(inside.sum()), sure[inside].mean().item(),
                         right[inside].mean().item()))
    return made

def ece(p, y):
    """기대 보정 오차(%). 구간마다 |맞힌 비율 - 평균 확신|을 장수의 비율로 평균한다."""
    return sum(n / len(y) * abs(hit - sure) for _, n, sure, hit in bins_of(p, y)) * 100

toy_p = torch.tensor([[0.9, 0.1]] * 5 + [[0.6, 0.4]] * 5)     # 예측 10개, 클래스 2개
toy_y = torch.tensor([0, 0, 0, 0, 1, 0, 0, 1, 1, 1])
print([(low, n, round(a, 2), round(b, 2)) for low, n, a, b in bins_of(toy_p, toy_y)])
print(round(ece(toy_p, toy_y), 2), round(accuracy(toy_p, toy_y), 1),
      round(nll(toy_p, toy_y), 4), round(brier(toy_p, toy_y), 4))

In [ ]:
track = []

def note(epoch, net):
    """에폭마다 학습 정확도와 평가용 5,000장의 정확도, 손실, 평균 확신, ECE를 적는다."""
    p = torch.softmax(scores_of(net, X_va[TEST]), dim=1)
    on_train = accuracy(torch.softmax(scores_of(net, X_tr), dim=1), y_tr)
    track.append((epoch, on_train, accuracy(p, y_va[TEST]), nll(p, y_va[TEST]),
                  p.max(dim=1)[0].mean().item() * 100, ece(p, y_va[TEST])))

lap = time.perf_counter()
torch.manual_seed(0)
fit(make_cnn(), X_tr, y_tr, 0, 40, watch=note)
took = {"40에폭 한 번": time.perf_counter() - lap}
track = np.array(track)
print("에폭   학습 정확도   정확도    손실   평균 확신     ECE")
for row in track[[0, 4, 9, 19, 29, 39]]:
    print(f"{int(row[0]):>3}      {row[1]:>6.2f}     {row[2]:>6.2f}  {row[3]:.4f}"
          f"    {row[4]:>6.2f}   {row[5]:>5.2f}")
print(int(track[track[:, 3].argmin(), 0]), int(track[track[:, 2].argmax(), 0]))

In [ ]:
lap = time.perf_counter()
nets = []
for seed in range(5):
    torch.manual_seed(seed)
    nets.append(fit(make_cnn(), X_tr, y_tr, seed, 30))
took["30에폭 모델 5개"] = time.perf_counter() - lap
Z = [scores_of(net, X_va) for net in nets]           # 시드마다 검증 10,000장의 점수
P = [torch.softmax(z, dim=1) for z in Z]

def report(name, p, y):
    wide = sum(ord(ch) > 0x1100 for ch in name)
    sure = p.max(dim=1)[0].mean().item() * 100
    print(name + " " * (22 - len(name) - wide) + f"{accuracy(p, y):>6.2f}  "
          f"{nll(p, y):.4f}  {brier(p, y):.4f}    {sure:>6.2f}   {ece(p, y):>5.2f}")

HEAD = "평가용 5,000장          정확도    손실  브라이어  평균 확신     ECE"
print(HEAD)
for seed in range(5):
    report(f"시드 {seed}", P[seed][TEST], y_va[TEST])
print("확신의 구간    장수   평균 확신   맞힌 비율   (시드 0)")
for low, n, sure, hit in bins_of(P[0][TEST], y_va[TEST]):
    print(f"{low:.1f}~{low + 0.1:.1f}     {n:>5}     {sure:.3f}      {hit:.3f}")

## 15.2 온도 조정: 순위는 그대로 두고 확률만 고치기

In [ ]:
TEMPS = torch.exp(torch.linspace(math.log(0.3), math.log(5), 400))   # 온도 후보 400개

def fit_T(z, y):
    """점수 z를 T로 나눈 뒤의 손실이 가장 작은 T를 후보 가운데 고른다."""
    losses = [F.cross_entropy(z / T, y).item() for T in TEMPS]
    return TEMPS[int(np.argmin(losses))].item()

logits = torch.tensor([[4.0, 2.0, 0.0]])
for T in [1.0, 2.0]:
    print(T, torch.softmax(logits / T, dim=1)[0].numpy().round(4))

curve = [F.cross_entropy(Z[0][FIT] / T, y_va[FIT]).item() for T in TEMPS]
T0 = fit_T(Z[0][FIT], y_va[FIT])
print(round(T0, 3), round(min(curve), 4),
      round(F.cross_entropy(Z[0][FIT], y_va[FIT]).item(), 4))

heat = [fit_T(z[FIT], y_va[FIT]) for z in Z]          # 시드마다 보정용으로 고른 온도
cooled = [torch.softmax(z / T, dim=1) for z, T in zip(Z, heat)]
print([round(T, 3) for T in heat])
print(HEAD)
for seed in range(5):
    report(f"시드 {seed}, 온도 조정", cooled[seed][TEST], y_va[TEST])
same = [(P[k].argmax(dim=1) == cooled[k].argmax(dim=1)).all().item() for k in range(5)]
print(same)
print("확신의 구간    장수   평균 확신   맞힌 비율   (시드 0, 온도 조정)")
for low, n, sure, hit in bins_of(cooled[0][TEST], y_va[TEST]):
    print(f"{low:.1f}~{low + 0.1:.1f}     {n:>5}     {sure:.3f}      {hit:.3f}")

In [ ]:
def bin_table(p, y, bins=10):
    """구간마다 맞힌 비율을 적은 표. 빈 구간에는 구간의 가운데 값을 적는다."""
    made = torch.tensor([(b + 0.5) / bins for b in range(bins)])
    for low, n, sure, hit in bins_of(p, y, bins):
        made[int(round(low * bins))] = hit
    return made

def binned_ece(p, y, fix, bins=10):
    """확신을 표의 값으로 바꾼 뒤의 ECE(%)."""
    sure, guess = p.max(dim=1)
    new = fix[torch.clamp((sure * bins).long(), max=bins - 1)]
    right = (guess == y).float()
    total = 0.0
    for b in range(bins):
        inside = torch.clamp((new * bins).long(), max=bins - 1) == b
        if inside.any():
            total += inside.float().mean().item() * abs(right[inside].mean().item()
                                                        - new[inside].mean().item())
    return total * 100

p0, y_test = P[0][TEST], y_va[TEST]
honest = binned_ece(p0, y_test, bin_table(P[0][FIT], y_va[FIT]))
peeked = binned_ece(p0, y_test, bin_table(p0, y_test))
print(round(ece(p0, y_test), 2), round(honest, 2), round(peeked, 2))

picker = torch.Generator().manual_seed(0)
print("장수    자기 자료로 고른 온도   보정용으로 고른 온도   손실의 차이")
gaps = {}
for n in [30, 100, 300, 1000, 5000]:
    found = []
    for _ in range(20):
        rows = torch.randperm(5000, generator=picker)[:n]
        z, y = Z[0][TEST][rows], y_test[rows]
        own = fit_T(z, y)                              # 평가할 이미지로 고른 온도
        found.append((own, F.cross_entropy(z / own, y).item()
                      - F.cross_entropy(z / T0, y).item()))
    found = np.array(found)
    gaps[n] = found[:, 1].mean()
    print(f"{n:>5}       {found[:, 0].min():.2f}~{found[:, 0].max():.2f}"
          f"              {T0:.2f}            {gaps[n]:+.4f}")

## 15.3 시드 앙상블

In [ ]:
together = torch.stack(P).mean(dim=0)                  # 다섯 모델의 확률을 평균한다
T_after = fit_T(torch.log(together[FIT]), y_va[FIT])  # 평균한 확률의 로그를 점수로 쓴다
after = torch.softmax(torch.log(together) / T_after, dim=1)
before = torch.stack(cooled).mean(dim=0)               # 온도 조정을 먼저 한 뒤에 평균
single = np.median([[accuracy(p[TEST], y_test), nll(p[TEST], y_test),
                     brier(p[TEST], y_test), p[TEST].max(dim=1)[0].mean().item() * 100,
                     ece(p[TEST], y_test)] for p in P], axis=0)
print(HEAD)
print("모델 하나 (중앙값)    " + f"{single[0]:>6.2f}  {single[1]:.4f}  {single[2]:.4f}"
      f"    {single[3]:>6.2f}   {single[4]:>5.2f}")
report("앙상블", together[TEST], y_test)
report("앙상블 뒤 온도 조정", after[TEST], y_test)
report("온도 조정 뒤 앙상블", before[TEST], y_test)
print(round(T_after, 3))
print("모델의 수    정확도     손실     ECE")
grow = []
for k in range(1, 6):
    part = torch.stack(P[:k]).mean(dim=0)[TEST]
    grow.append((accuracy(part, y_test), nll(part, y_test), ece(part, y_test)))
    print(f"{k:>5}      {grow[-1][0]:>6.2f}   {grow[-1][1]:.4f}   {grow[-1][2]:>5.2f}")
split = (torch.stack([p.argmax(dim=1) for p in P]) != P[0].argmax(dim=1)).any(dim=0)
print(round(split[TEST].float().mean().item() * 100, 2),
      round(accuracy(together[TEST][split[TEST]], y_test[split[TEST]]), 2),
      round(accuracy(together[TEST][~split[TEST]], y_test[~split[TEST]]), 2))

## 15.4 학습 자료에 없던 종류의 입력에 준 확률

In [ ]:
HIDDEN = [5, 8]                                        # 학습에서 뺄 클래스: 샌들, 가방
kept = [k for k in range(10) if k not in HIDDEN]
renumber = torch.full((10,), -1)
renumber[kept] = torch.arange(8)                       # 남은 여덟 클래스를 0~7로
known_tr = ~torch.isin(y_tr, torch.tensor(HIDDEN))
known_va = ~torch.isin(y_va, torch.tensor(HIDDEN))

lap = time.perf_counter()
narrow = []
for seed in range(5):
    torch.manual_seed(seed)
    narrow.append(fit(make_cnn(8), X_tr[known_tr], renumber[y_tr[known_tr]], seed, 30))
took["8클래스 모델 5개"] = time.perf_counter() - lap

noise = torch.rand(2000, 1, 28, 28, generator=torch.Generator().manual_seed(0))
noise = (noise - pixel_mean) / pixel_std               # 픽셀마다 0~1의 무작위 값
half = int(known_va.sum()) // 2                        # 본 클래스의 검증 자료를 반으로
groups = {"본 클래스": X_va[known_va][half:], "샌들": X_va[y_va == 5],
          "가방": X_va[y_va == 8], "무작위 픽셀": noise}
y_known = renumber[y_va[known_va]]
print(int(known_tr.sum()), int(known_va.sum()), {k: len(v) for k, v in groups.items()})
Zn = [{name: scores_of(net, x) for name, x in groups.items()} for net in narrow]
fit_part = scores_of(narrow[0], X_va[known_va][:half])
T8 = fit_T(fit_part, y_known[:half])                   # 앞 절반으로 고른 온도
print(round(accuracy(torch.softmax(Zn[0]["본 클래스"], dim=1), y_known[half:]), 2),
      round(T8, 3))

In [ ]:
def sureness(kind, name):
    """한 종류의 이미지마다 가장 큰 확률."""
    if kind == "모델 하나":
        p = torch.softmax(Zn[0][name], dim=1)
    elif kind == "온도 조정":
        p = torch.softmax(Zn[0][name] / T8, dim=1)
    else:
        p = torch.stack([torch.softmax(z[name], dim=1) for z in Zn]).mean(dim=0)
    return p.max(dim=1)[0]

def auc(positive, negative):
    """positive의 값이 negative의 값보다 클 확률(순위로 구한 AUC)."""
    both = torch.cat([positive, negative])
    rank = both.argsort().argsort().float() + 1
    n = len(positive)
    return ((rank[:n].sum() - n * (n + 1) / 2) / (n * len(negative))).item()

KINDS = ["모델 하나", "온도 조정", "앙상블"]
peak = {(kind, name): sureness(kind, name) for kind in KINDS for name in groups}
print("확률 0.9 이상을 준 비율(%)   본 클래스    샌들    가방   무작위 픽셀")
for kind in KINDS:
    wide = sum(ord(ch) > 0x1100 for ch in kind)
    share = [(peak[kind, name] >= 0.9).float().mean().item() * 100 for name in groups]
    print(kind + " " * (28 - len(kind) - wide) + "".join(f"{v:>8.1f}" for v in share))
print("가장 큰 확률의 중앙값")
for kind in KINDS:
    wide = sum(ord(ch) > 0x1100 for ch in kind)
    print(kind + " " * (28 - len(kind) - wide) + "".join(
        f"{peak[kind, name].median().item():>8.3f}" for name in groups))
said = {}
for name in ["샌들", "가방", "무작위 픽셀"]:
    counted = np.bincount(Zn[0][name].argmax(dim=1).numpy(), minlength=8)
    said[name] = [(names[kept[k]], int(counted[k])) for k in np.argsort(-counted)[:3]]
    print(name, said[name])
unseen = torch.cat([peak["모델 하나", "샌들"], peak["모델 하나", "가방"]])
print(round(auc(peak["모델 하나", "본 클래스"], unseen), 3),
      round(auc(peak["앙상블", "본 클래스"],
                torch.cat([peak["앙상블", "샌들"], peak["앙상블", "가방"]])), 3))

## 15.5 지표의 신뢰구간: 평가 자료의 변동과 시드의 변동

In [ ]:
def first_line(path):
    with open(path, encoding="cp949", errors="ignore") as f:
        return f.readline()

csvs = sorted(glob.glob("*.csv"))
power_files = [f for f in csvs if first_line(f).startswith("날짜,1시")]
weather_files = [f for f in csvs if first_line(f).startswith("지점,지점명")]
wide = pd.concat([pd.read_csv(f, encoding="cp949") for f in power_files])
long = wide.dropna(subset=["날짜"]).melt("날짜", var_name="시", value_name="MWh")
long["시각"] = pd.to_datetime(long["날짜"]) + pd.to_timedelta(
    long["시"].str.replace("시", "").astype(int), unit="h")
power = long.set_index("시각")["MWh"].sort_index() / 1000          # GW

FIVE = ["서울", "대전", "대구", "부산", "광주"]
w = pd.concat([pd.read_csv(f, encoding="cp949", usecols=["지점명", "일시", "기온(°C)"])
               for f in weather_files])
w["일시"] = pd.to_datetime(w["일시"])
temp = w[w["지점명"].isin(FIVE)].pivot(index="일시", columns="지점명",
                                      values="기온(°C)")
hours = pd.DataFrame({"GW": power, "기온": temp.interpolate().mean(axis=1)}).dropna()

HOLIDAYS = pd.to_datetime([
    "2023-01-01", "2023-01-21", "2023-01-22", "2023-01-23", "2023-01-24", "2023-03-01",
    "2023-05-05", "2023-05-27", "2023-05-29", "2023-06-06", "2023-08-15", "2023-09-28",
    "2023-09-29", "2023-09-30", "2023-10-02", "2023-10-03", "2023-10-09", "2023-12-25",
    "2025-01-01", "2025-01-27", "2025-01-28", "2025-01-29", "2025-01-30", "2025-03-01",
    "2025-03-03", "2025-05-05", "2025-05-06", "2025-06-03", "2025-06-06", "2025-08-15",
    "2025-10-03", "2025-10-05", "2025-10-06", "2025-10-07", "2025-10-08", "2025-10-09",
    "2025-12-25"])
day = (hours.index - pd.Timedelta(hours=1)).normalize()   # 24시는 그날의 마지막 한 시간
hours["시"] = (hours.index - pd.Timedelta(hours=1)).hour + 1
hours["요일"] = day.dayofweek                              # 월요일이 0, 일요일이 6
hours["월"] = day.month
hours["쉬는날"] = ((day.dayofweek >= 5) | day.isin(HOLIDAYS)).astype(int)
hours["날"] = day
hours["많음"] = (hours["GW"] >= 75).astype(int)            # 수요가 75GW 이상인 시간인가
h23, h25 = hours[day.year == 2023], hours[day.year == 2025]
y23, y25 = h23["많음"].to_numpy(), h25["많음"].to_numpy()
week = h23["날"].dt.isocalendar().week.to_numpy().astype(int)
held = week % 4 == 0                                      # 4주마다 한 주를 검증 자료로
print(len(y23), len(y25), round(float(y23.mean()), 4), round(float(y25.mean()), 4),
      int(held.sum()), h25["날"].nunique())

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

def log_loss(actual, prob):
    prob = np.clip(prob, 1e-12, 1 - 1e-12)
    return float(-(actual * np.log(prob) + (1 - actual) * np.log(1 - prob)).mean())

def brier2(actual, prob):
    return float(((prob - actual) ** 2).mean())

def auc2(actual, prob):
    return auc(torch.tensor(prob[actual == 1]), torch.tensor(prob[actual == 0]))

def added(df):
    """로지스틱 회귀의 특성: 기온의 꺾은 직선 둘, 쉬는 날, 시의 가변수 23개."""
    t = df["기온"].to_numpy()
    return np.column_stack([np.maximum(12 - t, 0), np.maximum(t - 20, 0), df["쉬는날"]]
                           + [(df["시"] == k).astype(float) for k in range(2, 25)])

FIVE_COLS = ["기온", "시", "요일", "월", "쉬는날"]
final = {}
logit = LogisticRegression(C=1e6, max_iter=5000).fit(added(h23), y23)
final["로지스틱 회귀"] = logit.predict_proba(added(h25))[:, 1]
boost = HistGradientBoostingClassifier(learning_rate=0.1, max_iter=73, max_depth=5,
                                       early_stopping=False, random_state=0)
final["부스팅"] = boost.fit(h23[FIVE_COLS], y23).predict_proba(h25[FIVE_COLS])[:, 1]

def exam(name, prob):
    wide = sum(ord(ch) > 0x1100 for ch in name)
    print(name + " " * (22 - len(name) - wide) + f"{auc2(y25, prob):.3f}    "
          f"{log_loss(y25, prob):.4f}     {brier2(y25, prob):.4f}")

print("2025년 8,736시간        AUC   로그 손실   브라이어")
for name, prob in final.items():
    exam(name, prob)

In [ ]:
def features(df, mean=None, std=None):
    """기온(표준화), 쉬는 날, 시·요일·월의 가변수. 모두 45열."""
    t = df["기온"].to_numpy()
    mean, std = (t.mean(), t.std()) if mean is None else (mean, std)
    cols = [(t - mean) / std, df["쉬는날"]]
    cols += [(df["시"] == k).astype(float) for k in range(1, 25)]
    cols += [(df["요일"] == k).astype(float) for k in range(7)]
    cols += [(df["월"] == k).astype(float) for k in range(1, 13)]
    return torch.tensor(np.column_stack(cols), dtype=torch.float32), mean, std

A23, t_mean, t_std = features(h23)
A25, _, _ = features(h25, t_mean, t_std)
b23 = torch.tensor(y23, dtype=torch.float32)
keep = torch.tensor(~held)

def fit_mlp(seed, epochs=20):
    """은닉층 둘(64, 64)인 신경망. 검증 주를 뺀 2023년으로 학습시킨다."""
    torch.manual_seed(seed)
    net = nn.Sequential(nn.Linear(45, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(),
                        nn.Linear(64, 1))
    optimizer = torch.optim.Adam(net.parameters(), lr=0.003)
    shuffler = torch.Generator().manual_seed(seed)
    x, y = A23[keep], b23[keep]
    for epoch in range(epochs):
        mixed = torch.randperm(len(x), generator=shuffler)
        for start in range(0, len(x), 64):
            rows = mixed[start:start + 64]
            optimizer.zero_grad()
            F.binary_cross_entropy_with_logits(net(x[rows])[:, 0], y[rows]).backward()
            optimizer.step()
    with torch.no_grad():
        return net(A23[~keep])[:, 0].numpy(), net(A25)[:, 0].numpy()

sigmoid = lambda z: 1 / (1 + np.exp(-np.clip(z, -50, 50)))
lap = time.perf_counter()
outs = [fit_mlp(seed) for seed in range(5)]            # (검증 주의 점수, 2025년의 점수)
took["전력수요 신경망 5개"] = time.perf_counter() - lap
grid = np.exp(np.linspace(np.log(0.3), np.log(8), 300))
warm = [grid[int(np.argmin([log_loss(y23[held], sigmoid(zv / T)) for T in grid]))]
        for zv, _ in outs]                             # 검증 주로 고른 온도
print(A23.shape, int(keep.sum()), [round(float(T), 2) for T in warm])
print("2025년 8,736시간        AUC   로그 손실   브라이어")
for seed, (zv, zt) in enumerate(outs):
    final[f"신경망 시드 {seed}"] = sigmoid(zt)
    final[f"온도 조정 시드 {seed}"] = sigmoid(zt / warm[seed])
    exam(f"신경망 시드 {seed}", final[f"신경망 시드 {seed}"])
for seed in range(5):
    exam(f"온도 조정 시드 {seed}", final[f"온도 조정 시드 {seed}"])
final["신경망 앙상블"] = np.mean([final[f"온도 조정 시드 {k}"] for k in range(5)],
                              axis=0)
exam("신경망 앙상블", final["신경망 앙상블"])
top = final["신경망 시드 0"] >= 0.9
print(int(top.sum()), round(float(final["신경망 시드 0"][top].mean()), 3),
      round(float(y25[top].mean()), 3))

In [ ]:
codes = pd.factorize(h25["날"])[0]                      # 시간마다 그 날의 번호
size = np.bincount(codes)                              # 날마다 시간의 수

def day_gap(one, other, draws=2000):
    """하루 단위로 복원 추출해서 두 모델의 브라이어 점수 차이의 95% 구간을 구한다."""
    apart = (np.bincount(codes, (final[one] - y25) ** 2)
             - np.bincount(codes, (final[other] - y25) ** 2))
    rng, found = np.random.default_rng(0), []
    for _ in range(draws):
        pick = rng.integers(0, len(size), len(size))
        found.append(apart[pick].sum() / size[pick].sum())
    return np.mean(found), np.quantile(found, 0.025), np.quantile(found, 0.975)

gap = {}
print("브라이어 점수의 차이 (음수이면 앞의 모델이 낫다), 95% 구간")
for kind in ["신경망", "온도 조정"]:
    for rival in ["로지스틱 회귀", "부스팅"]:
        for seed in range(5):
            mean, low, high = day_gap(f"{kind} 시드 {seed}", rival)
            gap[kind, rival, seed] = (mean, low, high)
        clear = sum(low > 0 or high < 0 for _, low, high in
                    [gap[kind, rival, seed] for seed in range(5)])
        print(f"{kind} - {rival}: 구간에 0이 없는 시드 {clear}개")
        for seed in range(5):
            mean, low, high = gap[kind, rival, seed]
            print(f"   시드 {seed}  {mean:+.4f}  [{low:+.4f}, {high:+.4f}]")

In [ ]:
loss = {name: np.bincount(codes, (prob - y25) ** 2) for name, prob in final.items()}

def seed_and_day(kind, rival, draws=2000):
    """날과 시드를 함께 복원 추출한다. 시드 다섯 개의 평균 브라이어에서 상대를 뺀다."""
    rng, found = np.random.default_rng(0), []
    for _ in range(draws):
        pick = rng.integers(0, len(size), len(size))
        seeds = rng.integers(0, 5, 5)
        ours = np.mean([loss[f"{kind} 시드 {k}"][pick].sum() for k in seeds])
        found.append((ours - loss[rival][pick].sum()) / size[pick].sum())
    return np.mean(found), np.quantile(found, 0.025), np.quantile(found, 0.975)

joint = {}
for kind in ["신경망", "온도 조정"]:
    for rival in ["로지스틱 회귀", "부스팅"]:
        joint[kind, rival] = seed_and_day(kind, rival)
        mean, low, high = joint[kind, rival]
        print(f"{kind} - {rival}: {mean:+.4f}  [{low:+.4f}, {high:+.4f}]")

apart = (final["로지스틱 회귀"] - y25) ** 2 - (final["부스팅"] - y25) ** 2
rng = np.random.default_rng(0)
by_hour = [apart[rng.integers(0, len(apart), len(apart))].mean() for _ in range(2000)]
by_day = day_gap("로지스틱 회귀", "부스팅")
print(f"[{np.quantile(by_hour, 0.025):+.4f}, {np.quantile(by_hour, 0.975):+.4f}]",
      f"[{by_day[1]:+.4f}, {by_day[2]:+.4f}]")
scores5 = [brier2(y25, final[f"신경망 시드 {k}"]) for k in range(5)]
print(round(min(scores5), 4), round(max(scores5), 4),
      round(float(np.std(scores5, ddof=1)), 4))

In [ ]:
import sys, platform, sklearn

print(sys.version.split()[0], np.__version__, pd.__version__, torch.__version__,
      sklearn.__version__)
print(platform.machine(), torch.get_default_dtype(), torch.get_num_threads())
for name, seconds in took.items():                           # 걸린 시간
    wide = sum(ord(ch) > 0x1100 for ch in name)
    print(name + " " * (24 - len(name) - wide) + f"{seconds:>5.0f}초")

In [ ]:
!pip -q install koreanize-matplotlib

In [ ]:
import matplotlib.pyplot as plt
import koreanize_matplotlib
from matplotlib.colors import to_rgba
from matplotlib.ticker import FuncFormatter

NAVY, BRICK, TEAL, GRAY = "#16324f", "#b5532a", "#2a7f73", "#9aa1ab"
BOX = {"fc": "white", "ec": "none", "alpha": 0.9, "pad": 1.5}


def frame(ax, title=None, color=NAVY, grid="y"):
    """그래프마다 반복하는 꾸밈: 제목, 보조선, 테두리 정리."""
    if title:
        ax.set_title(title, fontsize=9, loc="left", color=color, weight="bold")
    ax.grid(axis=grid, color="#e3e7ec", lw=0.6)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=8)
    ax.spines[["top", "right"]].set_visible(False)


def plain_ticks(axis):
    """로그 축의 눈금을 10의 거듭제곱 대신 보통 숫자로 적는다."""
    axis.set_major_formatter(FuncFormatter(lambda val, _: f"{val:,.0f}" if val >= 1 else f"{val:.7f}".rstrip("0")))

In [ ]:
# 그림 15-1
from matplotlib.patches import Patch, Rectangle
fig, axes = plt.subplots(3, 1, figsize=(7, 9.4), gridspec_kw={"height_ratios": [1, 1, 1.15]})
epochs1 = track[:, 0]
marks1 = [9, 29, 39]
ax = axes[0]
ax.plot(epochs1, track[:, 1], color=GRAY, lw=1.8, label="학습 이미지 6,000장")
ax.plot(epochs1, track[:, 2], color=NAVY, lw=1.8, label="평가용 5,000장")
for k in marks1:
    ax.text(epochs1[k], track[k, 1] + 1.0, f"{track[k, 1]:.1f}", ha="center", fontsize=9.4, color="#5b6470", weight="bold")
    ax.text(epochs1[k], track[k, 2] - 2.2, f"{track[k, 2]:.2f}", ha="center", fontsize=9.6, color=NAVY, weight="bold")
ax.set_xlim(0, 41.5); ax.set_ylim(76, 103); ax.set_ylabel("정확도 (%)", fontsize=9)
ax.legend(frameon=False, fontsize=9.2, loc="lower right", ncol=2)
frame(ax)
ax.set_title("① 정확도: 평가용 이미지의 정확도는 20에폭 뒤로 거의 그대로다", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(labelsize=9)
ax = axes[1]
ax.plot(epochs1, track[:, 3], color=BRICK, lw=1.8)
low1 = int(track[:, 3].argmin())
ax.scatter([epochs1[low1]], [track[low1, 3]], s=46, color="white", edgecolor=BRICK, lw=1.6, zorder=5)
ax.annotate(f"가장 낮은 곳: {int(epochs1[low1])}에폭, {track[low1, 3]:.4f}", xy=(epochs1[low1], track[low1, 3]), xytext=(19.5, 0.34), fontsize=9.2, color=BRICK, weight="bold",
            arrowprops=dict(arrowstyle="-|>", color=BRICK, lw=1.0))
for k in marks1:
    ax.text(epochs1[k], track[k, 3] + 0.022, f"{track[k, 3]:.4f}", ha="center", fontsize=9.6, color=BRICK, weight="bold")
ax.set_xlim(0, 41.5); ax.set_ylim(0.3, 0.62); ax.set_ylabel("평가용 이미지의 손실", fontsize=9)
frame(ax)
ax.set_title("② 손실: 정확도가 그대로인 동안 손실은 다시 올라간다", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(labelsize=9)
ax = axes[2]
ax.plot(epochs1, track[:, 4], color=BRICK, lw=1.8, label="평균 확신 (가장 큰 확률의 평균)")
ax.plot(epochs1, track[:, 2], color=NAVY, lw=1.8, label="정확도")
ax.fill_between(epochs1, track[:, 2], track[:, 4], where=track[:, 4] > track[:, 2], color=BRICK, alpha=0.13, lw=0)
for k in marks1:
    ax.text(epochs1[k] - (1.6 if k == 9 else 0), track[k, 4] + (1.5 if k == 9 else 0.7), f"{track[k, 4]:.2f}", ha="center", fontsize=9.6, color=BRICK, weight="bold")
    ax.text(epochs1[k], track[k, 2] - 1.5, f"{track[k, 2]:.2f}", ha="center", fontsize=9.6, color=NAVY, weight="bold")
    ax.text(epochs1[k], 78.6, f"ECE {track[k, 5]:.2f}", ha="center", fontsize=9.4, color="#111", weight="bold", bbox=BOX)
ax.text(34.5, 91.3, "칠한 폭이\n과신", ha="center", fontsize=9, color=BRICK, linespacing=1.4)
ax.set_xlim(0, 41.5); ax.set_ylim(76, 99); ax.set_xlabel("에폭", fontsize=9); ax.set_ylabel("%", fontsize=9)
ax.legend(frameon=False, fontsize=9.2, loc="upper left")
frame(ax)
ax.set_title("③ 평균 확신과 정확도: 오래 학습시킬수록 확신만 올라간다", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(labelsize=9)
fig.subplots_adjust(left=0.1, right=0.97, top=0.965, bottom=0.055, hspace=0.3)
fig.savefig("fig_15_1.png", dpi=300)
plt.show()
print(int(epochs1[low1]), round(float(track[low1, 3]), 4), [(int(epochs1[k]), round(float(track[k, 4] - track[k, 2]), 2)) for k in marks1])

In [ ]:
# 그림 15-2
def ladder(ax, rows, shade, title, upright=True):
    # 확신의 구간마다 맞힌 비율을 막대로 그리고 대각선과 비교한다
    ax.plot([0, 1], [0, 1], color="#666", lw=0.9, ls="--", zorder=1)
    for low, n, sure_, hit in rows:
        if n < 10:
            continue
        ax.bar(low + 0.05, hit, width=0.086, color=shade, zorder=2)
        ax.plot([low + 0.007, low + 0.093], [sure_, sure_], color="#111", lw=1.6, zorder=3)
        ax.text(low + 0.05, hit + 0.018 if hit >= sure_ else hit - 0.018, f"{hit:.2f}", ha="center", va="bottom" if hit >= sure_ else "top", fontsize=9.4,
                color=shade if hit >= sure_ else "white", weight="bold", zorder=4)
        ax.text(low + 0.05, 0.03, f"{n:,}장", ha="center", fontsize=9, color="white" if hit > 0.12 else "#111", rotation=90 if n >= 1000 or not upright else 0, va="bottom")
    ax.set_xlim(0.28, 1.01); ax.set_ylim(0, 1.06)
    ax.set_xticks(np.arange(0.3, 1.01, 0.1)); ax.set_xlabel("모델이 준 확신 (가장 큰 확률)", fontsize=9); ax.set_ylabel("실제로 맞힌 비율", fontsize=9)
    frame(ax)
    ax.set_title(title, fontsize=10, loc="left", color=NAVY, weight="bold")
    ax.tick_params(labelsize=9)

fig, axes = plt.subplots(2, 1, figsize=(7, 9.3))
rows2 = bins_of(P[0][TEST], y_va[TEST])
ladder(axes[0], rows2, BRICK, "① 30에폭 학습한 모델(시드 0): 막대가 검은 선보다 낮으면 과신")
worst2 = max((r for r in rows2 if r[1] >= 10), key=lambda r: r[2] - r[3])
axes[0].annotate(f"확신 {worst2[2]:.2f}에\n맞힌 비율 {worst2[3]:.2f}", xy=(worst2[0] + 0.05, worst2[3] + 0.09), xytext=(0.31, 0.74), fontsize=9.2, color="#111", linespacing=1.5,
                 arrowprops=dict(arrowstyle="-|>", color="#444", lw=1.0))
axes[0].text(0.3, 1.01, "검은 선: 그 구간의 평균 확신   점선: 확신과 맞힌 비율이 같은 선", fontsize=9, color="#444")
ax = axes[1]
counts2 = np.array([n for _, n, _, _ in rows2]); lows2 = np.array([low for low, _, _, _ in rows2])
ax.bar(lows2 + 0.05, counts2 / counts2.sum() * 100, width=0.086, color=NAVY)
for low, n in zip(lows2, counts2):
    ax.text(low + 0.05, n / counts2.sum() * 100 + 1.4, f"{n / counts2.sum() * 100:.1f}%", ha="center", fontsize=9.6, color=NAVY, weight="bold")
ax.set_xlim(0.18, 1.01); ax.set_ylim(0, 92); ax.set_xticks(np.arange(0.2, 1.01, 0.1))
ax.set_xlabel("모델이 준 확신", fontsize=9); ax.set_ylabel("평가용 5,000장 가운데 비율 (%)", fontsize=9)
ax.text(0.2, 80, f"평균 확신 {P[0][TEST].max(dim=1)[0].mean().item() * 100:.2f}%   정확도 {accuracy(P[0][TEST], y_va[TEST]):.2f}%   ECE {ece(P[0][TEST], y_va[TEST]):.2f}", fontsize=10, color="#111",
        weight="bold")
frame(ax)
ax.set_title("② 확신의 분포: 이미지의 대부분에 0.9 이상을 준다", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(labelsize=9)
fig.subplots_adjust(left=0.1, right=0.97, top=0.965, bottom=0.06, hspace=0.26)
fig.savefig("fig_15_2.png", dpi=300)
plt.show()
print((worst2[0], worst2[1], round(worst2[2], 3), round(worst2[3], 3)), [round(float(v), 1) for v in counts2 / counts2.sum() * 100])

In [ ]:
# 그림 15-4
from matplotlib.ticker import NullFormatter
fig = plt.figure(figsize=(7, 9.4))
grid = fig.add_gridspec(3, 2, height_ratios=[0.9, 1.25, 0.8], hspace=0.36, wspace=0.2, left=0.1, right=0.97, top=0.965, bottom=0.05)
at_one4 = F.cross_entropy(Z[0][FIT], y_va[FIT]).item()            # T = 1일 때의 손실
ax = fig.add_subplot(grid[0, :])
ax.plot(TEMPS.numpy(), curve, color=NAVY, lw=1.8)
ax.scatter([1.0, T0], [at_one4, min(curve)], s=46, color="white", edgecolor=BRICK, lw=1.6, zorder=5)
ax.annotate(f"T = 1 (조정 전)\n손실 {at_one4:.4f}", xy=(1.0, at_one4), xytext=(0.36, 0.56), fontsize=9.2, color="#111",
            linespacing=1.45, arrowprops=dict(arrowstyle="-|>", color="#444", lw=1.0))
ax.annotate(f"고른 온도 T = {T0:.3f}\n손실 {min(curve):.4f}", xy=(T0, min(curve)), xytext=(2.5, 0.52), fontsize=9.2, color=BRICK, weight="bold", linespacing=1.45,
            arrowprops=dict(arrowstyle="-|>", color=BRICK, lw=1.0))
ax.set_xscale("log"); ax.set_xticks([0.3, 0.5, 1, 2, 3, 5]); plain_ticks(ax.xaxis); ax.xaxis.set_minor_formatter(NullFormatter()); ax.set_xlim(0.3, 5); ax.set_ylim(0.33, 0.75)
ax.set_xlabel("온도 T (로그 축)", fontsize=9); ax.set_ylabel("보정용 5,000장의 손실", fontsize=9)
frame(ax)
ax.set_title("① 온도를 바꿔 가며 잰 보정용 이미지의 손실 (시드 0)", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(labelsize=9)
for col, (probs4, shade, title) in enumerate([(P[0], BRICK, "② 조정 전"), (cooled[0], TEAL, "③ 온도 조정 뒤")]):
    ax = fig.add_subplot(grid[1, col])
    ladder(ax, bins_of(probs4[TEST], y_va[TEST]), shade, f"{title}: ECE {ece(probs4[TEST], y_va[TEST]):.2f}", upright=False)
    ax.set_xlabel("모델이 준 확신", fontsize=9)
    if col:
        ax.set_ylabel("")
ax = fig.add_subplot(grid[2, :])
bars4 = [("조정 전", ece(P[0][TEST], y_va[TEST]), GRAY), ("온도 조정\n(보정용으로 고름)", ece(cooled[0][TEST], y_va[TEST]), TEAL), ("구간 표 보정\n(보정용으로 만듦)", honest, NAVY),
         ("구간 표 보정\n(평가용으로 만듦)", peeked, BRICK)]
for spot, (label, value, shade) in enumerate(bars4):
    ax.bar(spot, value, color=shade, width=0.56)
    ax.text(spot, value + 0.22, f"{value:.2f}", ha="center", fontsize=10.5, color="#111", weight="bold")
ax.annotate("평가할 이미지로 표를 만들면\nECE가 0이 된다", xy=(3, 1.0), xytext=(2.05, 4.3), fontsize=9.2, color=BRICK, linespacing=1.45, arrowprops=dict(arrowstyle="-|>", color=BRICK, lw=1.0))
ax.set_xticks(range(4)); ax.set_xticklabels([label for label, _, _ in bars4], fontsize=9.2, linespacing=1.4); ax.set_xlim(-0.5, 3.5); ax.set_ylim(0, 7.6)
ax.set_ylabel("평가용 5,000장의 ECE", fontsize=9)
frame(ax)
ax.set_title("④ 보정을 무엇으로 맞췄는가에 따른 ECE", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(axis="y", labelsize=9)
fig.savefig("fig_15_4.png", dpi=300)
plt.show()
print(round(at_one4, 4), round(min(curve), 4), [round(float(v), 2) for _, v, _ in bars4])

In [ ]:
# 그림 15-5
fig = plt.figure(figsize=(7, 9.4))
grid = fig.add_gridspec(3, 3, height_ratios=[1.1, 1, 0.62], hspace=0.42, wspace=0.4, left=0.1, right=0.97, top=0.955, bottom=0.05)
four5 = [("모델\n하나", single[[0, 1, 4]], GRAY), ("앙상블", [accuracy(together[TEST], y_test), nll(together[TEST], y_test), ece(together[TEST], y_test)], NAVY),
         ("앙상블\n뒤 온도", [accuracy(after[TEST], y_test), nll(after[TEST], y_test), ece(after[TEST], y_test)], TEAL),
         ("온도 뒤\n앙상블", [accuracy(before[TEST], y_test), nll(before[TEST], y_test), ece(before[TEST], y_test)], BRICK)]
for col, (what, lows5, form) in enumerate([("정확도 (%)", (86.5, 90.2), "{:.2f}"), ("손실", (0.28, 0.5), "{:.4f}"), ("ECE", (0, 6.9), "{:.2f}")]):
    ax = fig.add_subplot(grid[0, col])
    for spot, (label, values, shade) in enumerate(four5):
        ax.bar(spot, values[col], color=shade, width=0.66)
        ax.text(spot, values[col] + (lows5[1] - lows5[0]) * (0.085 if col == 1 and spot == 3 else 0.025), form.format(values[col]), ha="center", fontsize=9, color="#111", weight="bold")
    ax.set_xticks(range(4)); ax.set_xticklabels([label for label, _, _ in four5], fontsize=9, linespacing=1.3, rotation=0); ax.set_ylim(*lows5); ax.set_xlim(-0.6, 3.6)
    frame(ax)
    ax.set_title(("① " if col == 0 else "") + what + (" (높을수록 좋다)" if col == 0 else " (낮을수록 좋다)"), fontsize=9.6, loc="left", color=NAVY, weight="bold")
    ax.tick_params(axis="y", labelsize=9); ax.tick_params(axis="x", length=0)
ax = fig.add_subplot(grid[1, :])
sizes5 = np.arange(1, 6); grown5 = np.array(grow)
ax.plot(sizes5, grown5[:, 0], color=NAVY, marker="o", lw=1.8, ms=6.5)
for k in range(5):
    ax.text(sizes5[k], grown5[k, 0] + 0.16, f"{grown5[k, 0]:.2f}%", ha="center", fontsize=9.8, color=NAVY, weight="bold")
    ax.text(sizes5[k], 86.86, f"손실 {grown5[k, 1]:.4f}\nECE {grown5[k, 2]:.2f}", ha="center", fontsize=9, color=BRICK, linespacing=1.45)
ax.set_xticks(sizes5); ax.set_xticklabels([f"{k}개" for k in sizes5], fontsize=10); ax.set_xlim(0.5, 5.5); ax.set_ylim(86.6, 89.75)
ax.set_xlabel("확률을 평균한 모델의 수 (시드 0부터 차례로)", fontsize=9); ax.set_ylabel("정확도 (%)", fontsize=9)
frame(ax)
ax.set_title("② 모델을 하나씩 더할 때의 정확도, 손실, ECE", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(axis="y", labelsize=9)
ax = fig.add_subplot(grid[2, :])
apart5 = split[TEST].float().mean().item() * 100
for row, (label, share5, hit5, shade) in enumerate([("다섯 모델의 답이 모두 같은 이미지", 100 - apart5, accuracy(together[TEST][~split[TEST]], y_test[~split[TEST]]), NAVY),
                                                 ("답이 하나라도 다른 이미지", apart5, accuracy(together[TEST][split[TEST]], y_test[split[TEST]]), BRICK)]):
    ax.barh(row, hit5, color=shade, height=0.56)
    ax.text(hit5 + 1.2, row, f"앙상블 정확도 {hit5:.2f}%", va="center", fontsize=10.2, color="#111", weight="bold")
    ax.text(1.5, row, f"{label} ({share5:.2f}%)", va="center", fontsize=9.4, color="white", weight="bold")
ax.set_yticks([]); ax.invert_yaxis(); ax.set_xlim(0, 128); ax.set_xticks(range(0, 101, 20)); ax.set_xlabel("앙상블의 정확도 (%)", fontsize=9)
frame(ax, grid="x")
ax.set_title("③ 다섯 모델의 답이 갈리는 이미지에서 앙상블이 맞힌 비율", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(axis="x", labelsize=9)
fig.savefig("fig_15_5.png", dpi=300)
plt.show()
print([[round(float(v), 4) for v in values] for _, values, _ in four5], round(apart5, 2))

In [ ]:
# 그림 15-7
fig = plt.figure(figsize=(7, 9.4))
grid = fig.add_gridspec(3, 2, height_ratios=[1, 1, 1.25], hspace=0.44, wspace=0.16, left=0.1, right=0.97, top=0.955, bottom=0.085)
tones7 = {"본 클래스": NAVY, "샌들": BRICK, "가방": TEAL, "무작위 픽셀": GRAY}
notes7 = {"본 클래스": "학습에 쓴 여덟 클래스의 검증 이미지", "샌들": f"가장 많은 답: {said['샌들'][0][0]} {said['샌들'][0][1]}장, {said['샌들'][1][0]} {said['샌들'][1][1]}장",
          "가방": f"가장 많은 답: {said['가방'][0][0]} {said['가방'][0][1]}장, {said['가방'][1][0]} {said['가방'][1][1]}장",
          "무작위 픽셀": f"가장 많은 답: {said['무작위 픽셀'][0][0]} {said['무작위 픽셀'][0][1]:,}장"}
edges7 = np.linspace(0, 1, 21)
for spot, name in enumerate(groups):
    ax = fig.add_subplot(grid[spot // 2, spot % 2])
    values7 = peak["모델 하나", name].numpy()
    heights7 = np.histogram(values7, bins=edges7)[0] / len(values7) * 100
    ax.bar(edges7[:-1] + 0.025, heights7, width=0.044, color=tones7[name])
    over7 = (values7 >= 0.9).mean() * 100
    ax.axvline(0.9, color="#666", lw=0.8, ls="--")
    ax.text(0.88, 80, f"0.9 이상\n{over7:.1f}%", ha="right", fontsize=10.5, color=tones7[name] if name != "무작위 픽셀" else "#4a525d", weight="bold", linespacing=1.35)
    ax.text(0.02, 52, notes7[name], fontsize=9, color="#333")
    ax.set_xlim(0, 1.01); ax.set_ylim(0, 100); ax.set_xlabel("가장 큰 확률", fontsize=9)
    if spot % 2 == 0:
        ax.set_ylabel("이미지의 비율 (%)", fontsize=9)
    frame(ax)
    ax.set_title(f"{'①②③④'[spot]} {name} {len(values7):,}장" + (" (처음 보는 종류)" if name in ("샌들", "가방") else ""), fontsize=10, loc="left", color=NAVY, weight="bold")
    ax.tick_params(labelsize=9)
ax = fig.add_subplot(grid[2, :])
for spot, name in enumerate(groups):
    for shift, kind, look in [(-0.27, "모델 하나", 1.0), (0, "온도 조정", 0.62), (0.27, "앙상블", 0.32)]:
        value7 = (peak[kind, name] >= 0.9).float().mean().item() * 100
        ax.bar(spot + shift, value7, width=0.25, color=tones7[name], alpha=look)
        ax.text(spot + shift, value7 + 1.6, f"{value7:.1f}", ha="center", fontsize=9.8, color="#111", weight="bold")
        ax.text(spot + shift, -3.5, kind.replace(" ", "\n"), ha="center", va="top", fontsize=9, color="#444", linespacing=1.25)
ax.set_xticks(range(4)); ax.set_xticklabels(list(groups), fontsize=10.5, weight="bold"); ax.tick_params(axis="x", pad=38, length=0)
ax.set_xlim(-0.55, 3.55); ax.set_ylim(0, 112); ax.set_ylabel("확률 0.9 이상을 준 비율 (%)", fontsize=9)
frame(ax)
ax.set_title("⑤ 온도 조정과 앙상블을 쓴 뒤에 확률 0.9 이상을 준 비율", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(axis="y", labelsize=9)
fig.savefig("fig_15_7.png", dpi=300)
plt.show()
print([round(float((peak[kind, name] >= 0.9).float().mean()) * 100, 1) for kind in KINDS for name in groups])

In [ ]:
# 그림 15-9
fig, axes = plt.subplots(2, 1, figsize=(7, 9.3))
for ax, kind, mark9 in zip(axes, ["신경망", "온도 조정"], "①②"):
    for block, (rival, shade) in enumerate([("로지스틱 회귀", NAVY), ("부스팅", TEAL)]):
        for seed in range(5):
            mean9, low9, high9 = gap[kind, rival, seed]
            height = block * 8.0 + seed
            clear9 = low9 > 0 or high9 < 0
            ax.plot([low9, high9], [height, height], color=shade if clear9 else GRAY, lw=2.4)
            ax.scatter([mean9], [height], s=30, color="white", edgecolor=shade if clear9 else "#6b7480", lw=1.5, zorder=5)
            ax.text(-0.0265, height, f"시드 {seed}", va="center", fontsize=9.2, color="#111")
            ax.text(0.0325, height, f"{mean9:+.4f}".replace("-", "$-$") + ("" if clear9 else "  0을 포함"), va="center", ha="right", fontsize=9.2, color=shade if clear9 else "#6b7480",
                    weight="bold")
        mean9, low9, high9 = joint[kind, rival]
        height = block * 8.0 + 5.3
        ax.plot([low9, high9], [height, height], color=BRICK, lw=3.4)
        ax.scatter([mean9], [height], s=36, color="white", edgecolor=BRICK, lw=1.6, zorder=5)
        ax.text(-0.0265, height, "날과 시드를\n함께 뽑음", va="center", fontsize=9, color=BRICK, weight="bold", linespacing=1.25)
        ax.text(0.0325, height, f"[{low9:+.4f}, {high9:+.4f}]".replace("-", "$-$"), va="center", ha="right", fontsize=9.2, color=BRICK, weight="bold")
        clear_n = sum(gap[kind, rival, seed][1] > 0 or gap[kind, rival, seed][2] < 0 for seed in range(5))
        ax.text(-0.0265, block * 8.0 - 1.0, f"{kind} $-$ {rival}: 구간에 0이 없는 시드 {clear_n}개", fontsize=9.6, color=shade, weight="bold", bbox=BOX, zorder=6)
    ax.axvline(0, color="#333", lw=0.9)
    ax.set_ylim(14.2, -1.8); ax.set_xlim(-0.027, 0.033); ax.set_yticks([])
    ax.set_xlabel("2025년 브라이어 점수의 차이와 95% 구간 (왼쪽이면 신경망이 낫다)", fontsize=9)
    frame(ax, grid="x")
    ax.set_title(f"{mark9} " + ("학습한 그대로의 신경망" if kind == "신경망" else "온도 조정을 한 신경망") + "을 두 모델과 비교", fontsize=10, loc="left", color=NAVY, weight="bold")
    ax.tick_params(axis="x", labelsize=9)
fig.subplots_adjust(left=0.05, right=0.97, top=0.965, bottom=0.06, hspace=0.24)
fig.savefig("fig_15_9.png", dpi=300)
plt.show()
print({(kind, rival): [round(float(v), 4) for v in joint[kind, rival]] for kind in ["신경망", "온도 조정"] for rival in ["로지스틱 회귀", "부스팅"]})

In [ ]:
# 그림 15-10
def binary_bins(prob):
    # 예측 확률을 0.1 간격으로 나눠 (구간, 시간의 수, 평균 확률, 실제 비율)을 구한다
    where = np.clip((prob * 10).astype(int), 0, 9)
    return [(b / 10, int((where == b).sum()), float(prob[where == b].mean()), float(y25[where == b].mean())) for b in range(10) if (where == b).any()]

fig = plt.figure(figsize=(7, 9.4))
grid = fig.add_gridspec(2, 2, height_ratios=[1, 1.05], hspace=0.3, wspace=0.2, left=0.1, right=0.97, top=0.96, bottom=0.06)
for col, (key, shade, title) in enumerate([("신경망 시드 0", BRICK, "① 신경망(시드 0), 조정 전"), ("온도 조정 시드 0", TEAL, "② 온도 조정 뒤")]):
    ax = fig.add_subplot(grid[0, col])
    ax.plot([0, 1], [0, 1], color="#666", lw=0.9, ls="--")
    rows10 = binary_bins(final[key])
    ax.plot([r[2] for r in rows10], [r[3] for r in rows10], color=shade, marker="o", lw=1.8, ms=6)
    last10 = rows10[-1]
    ax.annotate(f"확률 {last10[2]:.2f}에\n실제 {last10[3]:.2f}\n({last10[1]:,}시간)", xy=(last10[2], last10[3]), xytext=(0.5, 0.2), fontsize=9.2, color="#111", linespacing=1.45,
                arrowprops=dict(arrowstyle="-|>", color="#444", lw=1.0))
    ax.text(0.03, 0.9, f"브라이어 {brier2(y25, final[key]):.4f}\n로그 손실 {log_loss(y25, final[key]):.4f}", fontsize=9.6, color=shade, weight="bold", linespacing=1.5)
    ax.set_xlim(0, 1.02); ax.set_ylim(0, 1.02); ax.set_xlabel("모델이 준 ‘75GW 이상’의 확률", fontsize=9)
    if col == 0:
        ax.set_ylabel("실제로 75GW 이상이었던 비율", fontsize=9)
    frame(ax)
    ax.set_title(title, fontsize=10, loc="left", color=NAVY, weight="bold")
    ax.tick_params(labelsize=9)
ax = fig.add_subplot(grid[1, :])
board10 = [("로지스틱 회귀", [brier2(y25, final["로지스틱 회귀"])], GRAY), ("신경망, 조정 전 (시드 5개)", [brier2(y25, final[f"신경망 시드 {k}"]) for k in range(5)], BRICK),
           ("신경망, 온도 조정 (시드 5개)", [brier2(y25, final[f"온도 조정 시드 {k}"]) for k in range(5)], TEAL), ("온도 조정한 신경망 5개의 앙상블", [brier2(y25, final["신경망 앙상블"])], NAVY),
           ("부스팅", [brier2(y25, final["부스팅"])], GRAY)]
for row, (label, values, shade) in enumerate(board10):
    ax.barh(row, np.median(values), color=shade, height=0.6)
    if len(values) > 1:
        ax.scatter(values, [row] * 5, s=20, color="white", edgecolor="#222", lw=0.7, zorder=5)
    ax.text(max(values) + 0.0006, row, f"{np.median(values):.4f}" + (f"  ({min(values):.4f}~{max(values):.4f})" if len(values) > 1 else ""), va="center", fontsize=10, color="#111", weight="bold")
ax.set_yticks(range(5)); ax.set_yticklabels([label for label, _, _ in board10], fontsize=9.4); ax.invert_yaxis(); ax.set_xlim(0.036, 0.0715)
ax.set_xlabel("2025년 8,736시간의 브라이어 점수 (낮을수록 좋다. 가로축은 0.036에서 시작)", fontsize=9)
frame(ax, grid="x")
ax.set_title("③ ‘75GW 이상인가’ 문제의 브라이어 점수", fontsize=10, loc="left", color=NAVY, weight="bold", x=-0.42)
ax.tick_params(axis="x", labelsize=9); ax.tick_params(axis="y", length=0)
pos = ax.get_position(); ax.set_position([0.33, pos.y0, 0.64, pos.height])
fig.savefig("fig_15_10.png", dpi=300)
plt.show()
print([(round(r[2], 3), round(r[3], 3), r[1]) for r in (binary_bins(final["신경망 시드 0"])[-1], binary_bins(final["온도 조정 시드 0"])[-1])], [round(float(np.median(v)), 4) for _, v, _ in board10])